# Solution: Implement Word2Vec Skip-Gram with Negative Sampling from Scratch

### Problem Statement
You are tasked with implementing the **Word2Vec Skip-Gram model** from scratch using PyTorch. The model should use **negative sampling** for efficient training. After training, you will visualize the learned word embeddings using **t-SNE** to observe semantic relationships between words.

### Requirements
1. **Data Acquisition**: Use a text corpus for training (we'll use a sample from a public dataset)
2. **Data Wrangling**: Preprocess text data, build vocabulary, and create training pairs
3. **Model Implementation**: Implement Skip-Gram architecture with negative sampling
4. **Training**: Train the model using appropriate loss function and optimizer
5. **Evaluation**: Visualize embeddings using t-SNE and analyze word relationships

### Constraints
- Implement the Skip-Gram model from scratch (not using pre-built Word2Vec libraries)
- Use negative sampling for training efficiency
- Train on a reasonable text corpus
- Visualize at least 50 words in the embedding space

## Step 1: Data Acquisition

**Data Source**: We will use the **Text8** dataset, a cleaned version of the first 100MB of Wikipedia text.
- Download from: http://mattmahoney.net/dc/text8.zip
- This is a single large text file with preprocessed Wikipedia articles
- Contains approximately 17 million words

In [ ]:
# Import required libraries
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from collections import Counter
import urllib.request
import zipfile
import os
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
from tqdm import tqdm

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

In [ ]:
# Download and load the text8 dataset
def download_text8():
    url = 'http://mattmahoney.net/dc/text8.zip'
    filename = 'text8.zip'
    
    if not os.path.exists('text8'):
        print('Downloading text8 dataset...')
        urllib.request.urlretrieve(url, filename)
        
        with zipfile.ZipFile(filename, 'r') as zip_ref:
            zip_ref.extractall('.')
        
        os.remove(filename)
        print('Download complete!')
    
    with open('text8', 'r') as f:
        text = f.read()
    
    return text

# Load text data
text = download_text8()
words = text.split()
print(f'Total words in corpus: {len(words)}')
print(f'Sample text: {" ".join(words[:50])}')

## Step 2: Data Wrangling

**Required Preprocessing Steps**:
1. Tokenize the text into words
2. Build vocabulary (map words to indices)
3. Remove rare words (appear less than min_count times)
4. Create context-target pairs for Skip-Gram (window-based sampling)
5. Implement subsampling of frequent words
6. Prepare negative sampling distribution

In [ ]:
# Build vocabulary and word-to-index mappings
class Vocabulary:
    def __init__(self, words, min_count=5):
        # Count word frequencies
        word_counts = Counter(words)
        
        # Filter words by minimum count
        self.word_counts = {word: count for word, count in word_counts.items() 
                           if count >= min_count}
        
        # Create word to index mapping
        self.word2idx = {word: idx for idx, word in enumerate(self.word_counts.keys())}
        self.idx2word = {idx: word for word, idx in self.word2idx.items()}
        
        self.vocab_size = len(self.word2idx)
        
        # Total word count
        self.total_words = sum(self.word_counts.values())
        
        print(f'Vocabulary size: {self.vocab_size}')
        print(f'Total words after filtering: {self.total_words}')
    
    def get_idx(self, word):
        return self.word2idx.get(word, None)
    
    def get_word(self, idx):
        return self.idx2word.get(idx, None)

# Build vocabulary
vocab = Vocabulary(words, min_count=5)

# Convert words to indices
word_indices = [vocab.get_idx(word) for word in words if vocab.get_idx(word) is not None]
print(f'Total word indices: {len(word_indices)}')

In [ ]:
# Implement subsampling for frequent words
def subsample_frequent_words(word_indices, vocab, threshold=1e-5):
    # Calculate word frequencies
    word_freqs = np.array([vocab.word_counts[vocab.get_word(idx)] / vocab.total_words 
                           for idx in word_indices])
    
    # Calculate keep probability
    keep_prob = (np.sqrt(word_freqs / threshold) + 1) * (threshold / word_freqs)
    keep_prob = np.minimum(keep_prob, 1.0)
    
    # Random sampling
    random_values = np.random.rand(len(word_indices))
    subsampled_indices = [idx for idx, (prob, rand) in enumerate(zip(keep_prob, random_values)) 
                         if rand < prob]
    
    return [word_indices[i] for i in subsampled_indices]

# Apply subsampling
word_indices = subsample_frequent_words(word_indices, vocab)
print(f'Words after subsampling: {len(word_indices)}')

In [ ]:
# Create training pairs (center word, context word)
def create_training_pairs(word_indices, window_size=5):
    pairs = []
    
    for idx, center_word in enumerate(word_indices):
        # Random window size
        w = np.random.randint(1, window_size + 1)
        
        # Get context words
        start = max(0, idx - w)
        end = min(len(word_indices), idx + w + 1)
        
        for context_idx in range(start, end):
            if context_idx != idx:
                pairs.append((center_word, word_indices[context_idx]))
    
    return pairs

# Create training pairs
training_pairs = create_training_pairs(word_indices, window_size=5)
print(f'Total training pairs: {len(training_pairs)}')
print(f'Sample pairs: {[(vocab.get_word(c), vocab.get_word(ctx)) for c, ctx in training_pairs[:5]]}')

In [ ]:
# Prepare negative sampling distribution
def create_negative_sampling_distribution(vocab, power=0.75):
    # Calculate word frequencies raised to power
    word_freqs = np.array([vocab.word_counts[vocab.get_word(idx)] 
                          for idx in range(vocab.vocab_size)])
    word_freqs = np.power(word_freqs, power)
    
    # Normalize to probabilities
    word_probs = word_freqs / np.sum(word_freqs)
    
    return word_probs

# Create negative sampling distribution
neg_sampling_probs = create_negative_sampling_distribution(vocab)
print(f'Negative sampling distribution shape: {neg_sampling_probs.shape}')

## Step 3: Model Implementation

**Model Architecture**: Skip-Gram with Negative Sampling
- **Input**: Center word (target word)
- **Output**: Context words prediction
- **Embedding Layer**: Maps words to dense vectors
- **Negative Sampling**: Sample k negative words per positive context word
- **Loss**: Binary cross-entropy for positive and negative samples

In [ ]:
# Implement Skip-Gram model with negative sampling
class SkipGramNegSampling(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super(SkipGramNegSampling, self).__init__()
        
        # Center word embeddings (input embeddings)
        self.center_embeddings = nn.Embedding(vocab_size, embedding_dim)
        
        # Context word embeddings (output embeddings)
        self.context_embeddings = nn.Embedding(vocab_size, embedding_dim)
        
        # Initialize embeddings
        self.center_embeddings.weight.data.uniform_(-0.5/embedding_dim, 0.5/embedding_dim)
        self.context_embeddings.weight.data.uniform_(-0.5/embedding_dim, 0.5/embedding_dim)
    
    def forward(self, center_words, context_words, negative_words):
        # Get embeddings
        center_embeds = self.center_embeddings(center_words)  # (batch_size, embedding_dim)
        context_embeds = self.context_embeddings(context_words)  # (batch_size, embedding_dim)
        negative_embeds = self.context_embeddings(negative_words)  # (batch_size, num_negative, embedding_dim)
        
        # Positive score (dot product)
        positive_score = torch.sum(center_embeds * context_embeds, dim=1)  # (batch_size,)
        positive_loss = -torch.log(torch.sigmoid(positive_score))
        
        # Negative score (batch matrix multiplication)
        negative_score = torch.bmm(negative_embeds, center_embeds.unsqueeze(2)).squeeze()  # (batch_size, num_negative)
        negative_loss = -torch.sum(torch.log(torch.sigmoid(-negative_score)), dim=1)
        
        # Total loss
        loss = torch.mean(positive_loss + negative_loss)
        
        return loss
    
    def get_embeddings(self):
        # Return center embeddings as final word embeddings
        return self.center_embeddings.weight.data.cpu().numpy()

print('Skip-Gram model with negative sampling defined!')

## Step 4: Training

**Training Configuration**:
- **Loss Function**: Binary Cross-Entropy Loss (implemented in model)
- **Optimizer**: Adam optimizer with learning rate 0.001
- **Hyperparameters**:
  - Embedding dimension: 100
  - Window size: 5
  - Negative samples: 5
  - Min word count: 5
  - Epochs: 5
  - Batch size: 512

In [ ]:
# Hyperparameters
EMBEDDING_DIM = 100
NUM_NEGATIVE = 5
BATCH_SIZE = 512
EPOCHS = 5
LEARNING_RATE = 0.001

# Initialize model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SkipGramNegSampling(vocab.vocab_size, EMBEDDING_DIM).to(device)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

print(f'Training on: {device}')
print(f'Model parameters: {sum(p.numel() for p in model.parameters())}')

In [ ]:
# Training loop
def train_word2vec(model, training_pairs, neg_sampling_probs, epochs, batch_size, num_negative):
    model.train()
    
    for epoch in range(epochs):
        total_loss = 0
        num_batches = 0
        
        # Shuffle training pairs
        np.random.shuffle(training_pairs)
        
        # Create batches
        for i in tqdm(range(0, len(training_pairs), batch_size), desc=f'Epoch {epoch+1}/{epochs}'):
            batch_pairs = training_pairs[i:i+batch_size]
            
            if len(batch_pairs) < batch_size:
                continue
            
            # Extract center and context words
            center_words = torch.LongTensor([pair[0] for pair in batch_pairs]).to(device)
            context_words = torch.LongTensor([pair[1] for pair in batch_pairs]).to(device)
            
            # Sample negative words
            negative_words = np.random.choice(
                vocab.vocab_size, 
                size=(batch_size, num_negative),
                p=neg_sampling_probs
            )
            negative_words = torch.LongTensor(negative_words).to(device)
            
            # Forward pass
            loss = model(center_words, context_words, negative_words)
            
            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            num_batches += 1
        
        avg_loss = total_loss / num_batches
        print(f'Epoch {epoch+1}/{epochs}, Average Loss: {avg_loss:.4f}')

# Train the model
train_word2vec(model, training_pairs, neg_sampling_probs, EPOCHS, BATCH_SIZE, NUM_NEGATIVE)

## Step 5: Evaluation and Visualization

**Evaluation Metrics**:
1. **t-SNE Visualization**: Reduce embeddings to 2D and visualize word clusters
2. **Similarity Analysis**: Find most similar words using cosine similarity
3. **Analogy Testing**: Test word analogies (e.g., king - man + woman ≈ queen)

In [ ]:
# Extract learned embeddings
embeddings = model.get_embeddings()
print(f'Embeddings shape: {embeddings.shape}')

In [ ]:
# Find most common words for visualization
most_common_words = [word for word, count in 
                     sorted(vocab.word_counts.items(), key=lambda x: x[1], reverse=True)[:100]]
most_common_indices = [vocab.get_idx(word) for word in most_common_words]

# Get embeddings for most common words
common_embeddings = embeddings[most_common_indices]

# Apply t-SNE
tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
embeddings_2d = tsne.fit_transform(common_embeddings)

# Visualize embeddings
plt.figure(figsize=(16, 12))
plt.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], alpha=0.5)

# Annotate with word labels
for i, word in enumerate(most_common_words):
    plt.annotate(word, (embeddings_2d[i, 0], embeddings_2d[i, 1]), 
                fontsize=9, alpha=0.7)

plt.title('Word2Vec Embeddings Visualization (t-SNE)', fontsize=16)
plt.xlabel('t-SNE Dimension 1')
plt.ylabel('t-SNE Dimension 2')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Find similar words using cosine similarity
def cosine_similarity(vec1, vec2):
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))

def find_similar_words(word, embeddings, vocab, top_k=10):
    word_idx = vocab.get_idx(word)
    if word_idx is None:
        print(f'Word "{word}" not in vocabulary')
        return
    
    word_embedding = embeddings[word_idx]
    
    # Calculate similarities
    similarities = []
    for idx in range(vocab.vocab_size):
        if idx != word_idx:
            sim = cosine_similarity(word_embedding, embeddings[idx])
            similarities.append((vocab.get_word(idx), sim))
    
    # Sort by similarity
    similarities.sort(key=lambda x: x[1], reverse=True)
    
    print(f'\nMost similar words to "{word}":')
    for similar_word, sim in similarities[:top_k]:
        print(f'  {similar_word}: {sim:.4f}')

# Test similarity
test_words = ['king', 'computer', 'woman', 'france', 'science']
for word in test_words:
    find_similar_words(word, embeddings, vocab, top_k=5)

In [ ]:
# Test word analogies
def word_analogy(word_a, word_b, word_c, embeddings, vocab, top_k=5):
    # Get word indices
    idx_a = vocab.get_idx(word_a)
    idx_b = vocab.get_idx(word_b)
    idx_c = vocab.get_idx(word_c)
    
    if None in [idx_a, idx_b, idx_c]:
        print('One or more words not in vocabulary')
        return
    
    # Calculate analogy vector: vec(b) - vec(a) + vec(c)
    analogy_vec = embeddings[idx_b] - embeddings[idx_a] + embeddings[idx_c]
    
    # Find most similar words
    similarities = []
    for idx in range(vocab.vocab_size):
        if idx not in [idx_a, idx_b, idx_c]:
            sim = cosine_similarity(analogy_vec, embeddings[idx])
            similarities.append((vocab.get_word(idx), sim))
    
    similarities.sort(key=lambda x: x[1], reverse=True)
    
    print(f'\nAnalogy: "{word_a}" is to "{word_b}" as "{word_c}" is to:')
    for word, sim in similarities[:top_k]:
        print(f'  {word}: {sim:.4f}')

# Test analogies
print('Testing word analogies:')
word_analogy('man', 'king', 'woman', embeddings, vocab)
word_analogy('paris', 'france', 'berlin', embeddings, vocab)
word_analogy('good', 'better', 'bad', embeddings, vocab)

In [ ]:
# Visualize specific word clusters
def visualize_word_clusters(word_groups, embeddings, vocab):
    colors = ['red', 'blue', 'green', 'purple', 'orange', 'brown', 'pink', 'gray']
    
    plt.figure(figsize=(14, 10))
    
    all_words = []
    all_colors = []
    
    for i, (group_name, words) in enumerate(word_groups.items()):
        # Get embeddings for words in this group
        group_indices = [vocab.get_idx(word) for word in words if vocab.get_idx(word) is not None]
        if not group_indices:
            continue
        
        group_embeddings = embeddings[group_indices]
        valid_words = [words[j] for j, idx in enumerate([vocab.get_idx(w) for w in words]) if idx is not None]
        
        all_words.extend(valid_words)
        all_colors.extend([colors[i % len(colors)]] * len(valid_words))
    
    # Get all embeddings
    all_indices = [vocab.get_idx(word) for word in all_words]
    all_embeddings = embeddings[all_indices]
    
    # Apply t-SNE
    tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, len(all_words)-1))
    embeddings_2d = tsne.fit_transform(all_embeddings)
    
    # Plot
    for i, word in enumerate(all_words):
        plt.scatter(embeddings_2d[i, 0], embeddings_2d[i, 1], 
                   c=all_colors[i], s=100, alpha=0.6)
        plt.annotate(word, (embeddings_2d[i, 0], embeddings_2d[i, 1]), 
                    fontsize=10, alpha=0.8)
    
    plt.title('Word Clusters Visualization', fontsize=16)
    plt.xlabel('t-SNE Dimension 1')
    plt.ylabel('t-SNE Dimension 2')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

# Define word groups for visualization
word_groups = {
    'Countries': ['france', 'germany', 'italy', 'spain', 'england', 'america'],
    'Numbers': ['one', 'two', 'three', 'four', 'five', 'six', 'seven', 'eight'],
    'Family': ['mother', 'father', 'sister', 'brother', 'son', 'daughter'],
    'Time': ['day', 'week', 'month', 'year', 'morning', 'evening'],
    'Colors': ['red', 'blue', 'green', 'yellow', 'black', 'white']
}

visualize_word_clusters(word_groups, embeddings, vocab)